# 30 — Validate the core T-Box

Checks the two deliverables `20_generate.ipynb` produced. Four checks, all
fail-fast:

1. **Baselines** — triples and entity counts match the recorded numbers for the
   pinned commit. A deviation means either a source change (document the delta)
   or a generation bug (investigate).
2. **Namespace well-formedness** — the regression guard for the repair in
   `docs/known-gaps.md` §1a. Every IRI in a schema's own namespace must carry a
   separator. If the patch stops being applied, this fails.
3. **Graph separation** — the two graphs share no domain IRIs, only standard
   vocabulary. This is what decision D1 bought.
4. **Collision separation** — each of the six names declared in both published
   schemas resolves to two distinct IRIs, one per graph, and both are present.
5. **Decision D7 holds** — the ontology is named under w3id, no term is, and the
   advertised term namespace is still CDISC's.
6. **The JSON-LD contexts** parse, carry an explicit `@version`, and map
   `conceptId` to `@id`.

Also writes two CSV reports to `../reports/`.

## Baselines — pinned commit `031429b1`, package date 2026-07-14

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
BUILD     = "../build"
REPORTS   = "../reports"

BC_TTL   = "cosmos_bc_v1.ttl"
SDTM_TTL = "cosmos_sdtm_v1.ttl"

BC_CONTEXT   = "cosmos_bc_v1.context.jsonld"
SDTM_CONTEXT = "cosmos_sdtm_v1.context.jsonld"

CONTEXT_TERMS = {BC_CONTEXT: 27, SDTM_CONTEXT: 56}
JSONLD_VERSION = 1.1

BC_NS   = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0"
SDTM_NS = "https://www.cdisc.org/cosmos/sdtm_v1.0"

# Decision D7, option A: the ontology is named under w3id, the terms are not.
W3ID = "https://w3id.org/cdisc/cosmos/"
VERSION = "0.1.0"

ONTOLOGY_IRI = {
    BC_TTL: W3ID + "bc/",
    SDTM_TTL: W3ID + "sdtm/",
}

# Third copy of these numbers, after 20_generate's printout and README.md.
# When the pin is bumped and they drift, update all three.
BASELINES = {
    BC_TTL: {
        "triples": 535,
        "top_level_classes": 6,
        "permissible_values": 15,
        "object_properties": 5,
        "datatype_properties": 13,
    },
    SDTM_TTL: {
        "triples": 2028,
        "top_level_classes": 24,
        "permissible_values": 146,
        "object_properties": 13,
        "datatype_properties": 30,
    },
}

# Declared in both published schemas. docs/known-gaps.md 1.
COLLIDING_SLOTS = [
    "conceptId",
    "dataType",
    "href",
    "packageDate",
    "packageType",
    "shortName",
]

failures = []


def check(name, actual, expected):
    if actual == expected:
        print(f"ok    {name}: {actual}")
    else:
        print(f"FAIL  {name}: expected {expected}, got {actual}")
        failures.append(name)

## 1 — Parse and check baselines

In [ ]:
from pathlib import Path

from rdflib import Graph, URIRef
from rdflib.namespace import OWL, RDF

graphs = {}

for target, expected in BASELINES.items():
    g = Graph().parse(str(Path(ROOT, target)), format="turtle")
    graphs[target] = g

    classes = {str(s) for s in g.subjects(RDF.type, OWL.Class)}
    pvs = {c for c in classes if "#" in c}

    check(f"{target} triples", len(g), expected["triples"])
    check(f"{target} top-level owl:Class", len(classes) - len(pvs), expected["top_level_classes"])
    check(f"{target} permissible values", len(pvs), expected["permissible_values"])
    check(
        f"{target} owl:ObjectProperty",
        len(set(g.subjects(RDF.type, OWL.ObjectProperty))),
        expected["object_properties"],
    )
    check(
        f"{target} owl:DatatypeProperty",
        len(set(g.subjects(RDF.type, OWL.DatatypeProperty))),
        expected["datatype_properties"],
    )
    print()

## 2 — Namespace well-formedness

The regression guard for `docs/known-gaps.md` §1a. Every IRI inside a schema's own
namespace must continue with a separator, so a term IRI reads
`…/biomedical_concept_v1.0/BiomedicalConcept` and not
`…/biomedical_concept_v1.0BiomedicalConcept`.

Two IRIs are exempt because they are the ontology resource itself rather than
terms in the namespace: the bare schema id, and the schema id plus the
generator's `.owl.ttl` ontology-URI suffix.

In [ ]:
for target, namespace in ((BC_TTL, BC_NS), (SDTM_TTL, SDTM_NS)):
    g = graphs[target]
    exempt = {namespace, namespace + ".owl.ttl"}

    in_namespace = set()
    for triple in g:
        for node in triple:
            if isinstance(node, URIRef) and str(node).startswith(namespace):
                in_namespace.add(str(node))

    malformed = sorted(
        iri
        for iri in in_namespace - exempt
        if not iri.startswith(namespace + "/")
    )
    check(f"{target} malformed IRIs", malformed, [])
    print(f"      {len(in_namespace - exempt)} term IRIs in {namespace}/")
    print()

## 3 — Graph separation

Neither deliverable may contain an IRI from the other's COSMoS namespace. That is
the claim decision D1 rests on, stated directly rather than as an allowlist of
"standard" vocabularies — an allowlist would need extending every time the header
gains a predicate, and would eventually pass by being edited rather than by being
true.

What the two graphs *do* share is shared vocabulary: RDF, RDFS, OWL, XSD, SKOS,
Dublin Core, VANN, the license IRI. That count is printed for information.

In [ ]:
def iris(g):
    found = set()
    for triple in g:
        for node in triple:
            if isinstance(node, URIRef):
                found.add(str(node))
    return found


bc_iris = iris(graphs[BC_TTL])
sdtm_iris = iris(graphs[SDTM_TTL])

check(
    f"{BC_TTL} free of SDTM-namespace IRIs",
    sorted(i for i in bc_iris if i.startswith(SDTM_NS)),
    [],
)
check(
    f"{SDTM_TTL} free of BC-namespace IRIs",
    sorted(i for i in sdtm_iris if i.startswith(BC_NS)),
    [],
)
print(f"      {len(bc_iris & sdtm_iris)} IRIs shared, all of them shared vocabulary")

## 4 — Collision separation, and the report that documents it

Each of the six names declared in both published schemas must appear as a term in
**both** graphs, at two **distinct** IRIs. That is the concrete thing decision D1
bought: the two meanings stay apart instead of one silently winning.

The report also carries each side's pattern and range, read from the models
themselves — the evidence table for the upstream issue.

In [ ]:
import csv

from linkml_runtime.utils.schemaview import SchemaView

bc_view = SchemaView(str(Path(BUILD, "cosmos_bc_model.patched.yaml")))
sdtm_view = SchemaView(str(Path(DOWNLOADS, "cosmos_sdtm_model.yaml")))

rows = []
for name in COLLIDING_SLOTS:
    bc_iri = f"{BC_NS}/{name}"
    sdtm_iri = f"{SDTM_NS}/{name}"

    if bc_iri not in bc_iris:
        check(f"{name} present in {BC_TTL}", False, True)
    if sdtm_iri not in sdtm_iris:
        check(f"{name} present in {SDTM_TTL}", False, True)
    if bc_iri == sdtm_iri:
        check(f"{name} IRIs distinct", False, True)

    bc_slot = bc_view.get_slot(name)
    sdtm_slot = sdtm_view.get_slot(name)
    rows.append(
        {
            "slot": name,
            "bc_iri": bc_iri,
            "bc_range": bc_slot.range,
            "bc_pattern": bc_slot.pattern or "",
            "bc_description": bc_slot.description or "",
            "sdtm_iri": sdtm_iri,
            "sdtm_range": sdtm_slot.range,
            "sdtm_pattern": sdtm_slot.pattern or "",
            "sdtm_description": sdtm_slot.description or "",
        }
    )

check("colliding slots separated across the two graphs", len(rows), len(COLLIDING_SLOTS))

Path(REPORTS).mkdir(parents=True, exist_ok=True)
report = Path(REPORTS, "colliding_slots.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"      wrote {report}")

for row in rows:
    print(f"      {row['slot']:12s} bc[{row['bc_range']}, {row['bc_pattern'] or 'no pattern'}]"
          f"  vs  sdtm[{row['sdtm_range']}, {row['sdtm_pattern'] or 'no pattern'}]")

## 5 — Decision D7 stays decision D7

The guard against option A quietly drifting into option B. Three things must hold:

- each graph's ontology IRI is the w3id one, with a matching bare-numeric
  `owl:versionIRI`;
- **no term** sits in the w3id namespace — the only two IRIs there are the
  ontology and its version;
- the term namespace the ontology advertises as `vann:preferredNamespaceUri` is
  CDISC's, not this repo's.

If a later change starts minting terms under w3id, this fails.

In [ ]:
from rdflib import URIRef
from rdflib.namespace import OWL, RDF

VANN_PREFERRED_NAMESPACE = URIRef("http://purl.org/vocab/vann/preferredNamespaceUri")

for target, namespace in ((BC_TTL, BC_NS), (SDTM_TTL, SDTM_NS)):
    g = graphs[target]
    expected_iri = ONTOLOGY_IRI[target]

    ontologies = [str(s) for s in g.subjects(RDF.type, OWL.Ontology)]
    check(f"{target} ontology IRI", ontologies, [expected_iri])

    version_iris = [str(o) for o in g.objects(URIRef(expected_iri), OWL.versionIRI)]
    check(f"{target} owl:versionIRI", version_iris, [expected_iri + VERSION])

    preferred = [str(o) for o in g.objects(URIRef(expected_iri), VANN_PREFERRED_NAMESPACE)]
    check(f"{target} advertises CDISC term namespace", preferred, [namespace + "/"])

    in_w3id = set()
    for triple in g:
        for node in triple:
            if isinstance(node, URIRef) and str(node).startswith(W3ID):
                in_w3id.add(str(node))
    check(
        f"{target} IRIs in the w3id namespace",
        sorted(in_w3id),
        sorted({expected_iri, expected_iri + VERSION}),
    )
    print()

## 6 — JSON-LD contexts

Shape checks on the P2 deliverables. `conceptId` mapping to `@id` is the hinge
decision D2 turns on, so it is asserted rather than printed.

In [ ]:
import json

for target, expected_terms in CONTEXT_TERMS.items():
    document = json.loads(Path(ROOT, target).read_text(encoding="utf-8"))

    check(f"{target} top-level keys", sorted(document), ["@context"])
    context = document["@context"]
    check(f"{target} @version", context.get("@version"), JSONLD_VERSION)
    check(f"{target} conceptId mapping", context.get("conceptId"), "@id")
    check(f"{target} term count", len(context) - 1, expected_terms)
    print()

## Top-level class inventory

In [ ]:
from rdflib.namespace import RDFS

rows = []
for target, namespace in ((BC_TTL, BC_NS), (SDTM_TTL, SDTM_NS)):
    g = graphs[target]
    for subject in sorted(g.subjects(RDF.type, OWL.Class), key=str):
        iri = str(subject)
        if "#" in iri:
            continue
        labels = [str(o) for o in g.objects(subject, RDFS.label)]
        rows.append(
            {
                "graph": target,
                "iri": iri,
                "local_name": iri.rsplit("/", 1)[-1],
                "label": labels[0] if labels else "",
                "in_schema_namespace": iri.startswith(namespace + "/"),
            }
        )

report = Path(REPORTS, "top_level_classes.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"wrote {report}: {len(rows)} rows")
for row in rows:
    if not row["in_schema_namespace"]:
        print(f"   outside own namespace: {row['graph']:20s} {row['iri']}")

## Result

In [ ]:
if failures:
    raise RuntimeError(f"{len(failures)} check(s) failed: {', '.join(failures)}")
print("All core T-Box checks passed.")